# CodeGen — LoRA sweep: train several adapters in ONE Kaggle commit (Unsloth, T4)

Same recipe as `codegen_finetune.ipynb` (same install, data, model, prompt masking, trainer settings). The only
difference: section 4 lists several configs in `SWEEP`, and section 6 trains them one after another, each from a
fresh copy of the base model, and saves each adapter to `/kaggle/working/<RUN_NAME>/` as soon as it finishes.

- A run that crashes is recorded and skipped; the runs after it still happen, and finished adapters are kept.
- Checkpoint folders are deleted after a run succeeds, so the output stays small.
- `sweep_summary.json` at the end lists every run with its train loss and dev loss.

**How to run it overnight (Kaggle)**
1. Settings: **GPU T4 x2**, **Internet On**. Add Input → the `approved_v2_1` dataset.
2. Smoke test (optional, ~10 min): set `SMOKE_TEST = True`, Run All interactively. Every config trains for 5 steps.
   Then set it back to `False`.
3. **Save Version → Save & Run All (Commit)**. You can close the tab: a commit runs in the background (12 h limit).
   With the default 4 configs, expect about 3–3.5 h.
4. Next day: open the version → Output tab; you should see one folder per `RUN_NAME`.
   For the eval notebook, Add Input → Notebook Output → this version (all adapters come in one input).

Not a benchmark: dev loss only says the run worked. Accuracy comes from `codegen_finetune_eval.ipynb` + Atlas.

## 0. Environment (run first, before anything imports torch)

In [ ]:
import os
import sys

# setting these env variables BEFORE anything imports torch, otherwise they dont do anything
# kaggle gives 2 T4s but unsloth only trains on 1, so just use gpu 0
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
# kaggle has wandb preinstalled and the trainer keeps asking for an api key, so turning it off
os.environ["WANDB_DISABLED"] = "true"
# transformers tries to import kaggle's tensorflow for no reason, this stops it
os.environ["USE_TF"] = "0"
# just stops the tokenizers fork warning spam
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# figure out if we're on kaggle or colab (kaggle always has /kaggle/working)
ON_KAGGLE = os.path.exists("/kaggle/working")
ON_COLAB = False
if "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ:
    ON_COLAB = True

if ON_KAGGLE:
    platform = "kaggle"
elif ON_COLAB:
    platform = "colab"
else:
    platform = "other"
python_version = sys.version.split()[0]
print("platform:", platform, "| python", python_version)

## 1. Config

In [ ]:
# ======================= CONFIG =======================
# only change stuff in here. everything below just reads these values

# ---- data ----
# the files from the approved_v2_1_mql0010_view dataset (MQL-0010 holdout removed).
# sweep: sft_train (90 build DBs). final model: set TRAIN_FILE = "sft_full.prompt_completion.jsonl"
# train = what the model learns from, dev = held out (different databases) to check it
TRAIN_FILE = "sft_train.prompt_completion.jsonl"
DEV_FILE = "sft_dev.prompt_completion.jsonl"
# leave it None and it finds the files by itself
# (kaggle: searches /kaggle/input, colab: uses COLAB_DRIVE_DIR)
DATA_DIR = None
COLAB_USE_DRIVE = True   # colab only. set False if you want to upload the 2 files manually instead
COLAB_DRIVE_DIR = "/content/drive/MyDrive/codegen_finetune/approved_v2_1"

# ---- install ----
# right now this just installs whatever the latest unsloth is
# once a run works, check pip_freeze.txt in the output and pin the exact versions here
# so everyone in the team gets the same setup, like:
# UNSLOTH_SPEC = ["unsloth==<version>", "unsloth_zoo==<version>"]
UNSLOTH_SPEC = ["unsloth"]

# ---- model ----
# 3B coder model that unsloth already squished down to 4-bit
# 4-bit = every weight stored in 4 bits instead of 16. kinda like saving a photo as jpeg instead of raw,
# tiny bit of quality loss but ~4x smaller, which is the only reason it fits on a free T4
# want to try 7B? use "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit" (smarter, but slower and barely fits on a T4)
MODEL_NAME = "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit"
# max tokens in ONE example (system prompt + schema + question + answer all together)
# think of it as the page limit on an answer sheet, anything longer gets dropped
# bigger = more gpu memory. our longest example is ~1.4k tokens (section 7 prints it) so 2048 is safe
MAX_SEQ_LENGTH = 2048

# ---- LoRA ----
# LoRA in one line: we DONT touch the original 3B weights at all (they're frozen),
# we just add small extra matrices next to some layers and only train those.
# analogy: the base model is a printed textbook you cant edit, LoRA is the sticky notes you put on top.

# r (rank) = how big each sticky note is
# bigger r = more room to learn new stuff, but more params, slower, and easier to overfit (just memorise)
# r=16 gives ~30M trainable params, less than 1% of the model. 8 and 32 are also normal values to try
LORA_R = 16

# alpha = how "loud" the sticky notes are compared to the original book
# whatever LoRA learns gets multiplied by alpha / r before it's added, so here it's 32 / 16 = 2x
# too high = the notes drown out what the model already knows, too low = they barely do anything
# project plan says keep alpha = 2 * r, so if you change r change this too (r=8 -> 16, r=32 -> 64)
LORA_ALPHA = 32

# dropout = randomly switch off some % of the LoRA during training so it cant just memorise
# like practicing with some of your notes randomly covered, so you actually learn the thing
# 0 here bcoz unsloth is fastest at 0, and with 1 epoch overfitting isnt a big problem anyway
LORA_DROPOUT = 0.0

# which layers get sticky notes
# q/k/v/o_proj = attention (the part that decides which words to "look at")
# gate/up/down_proj = MLP (the part that does the actual "thinking" on what it looked at)
# all 7 is the usual full set. removing some = fewer params but usually worse results
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# ---- training ----
# epoch = one full pass over all the training examples. like reading the whole textbook once
# 1 is enough for now. 2 can help but it also starts memorising
NUM_EPOCHS = 1
# set this to like 30 for a quick test run to check nothing crashes. -1 = ignore it and use NUM_EPOCHS
MAX_STEPS = -1

# batch size = how many examples the gpu processes together in one go
# grad accum = how many of those batches we add up before actually updating the weights
# so effective batch = 2 * 4 = 8
# analogy: teacher wants to check 8 papers before giving feedback, but her desk only fits 2,
# so she reads 2, then 2 more... 4 times, and THEN gives feedback. same result, less desk space (gpu memory)
BATCH_SIZE = 2
GRAD_ACCUM = 4

# learning rate = how big a step the model takes each time it updates
# like walking downhill blindfolded: big steps = fast but you overshoot the bottom,
# tiny steps = safe but takes forever. 2e-4 (= 0.0002) is the usual LoRA value
LEARNING_RATE = 2e-4
# warmup = for the first 3% of steps the learning rate slowly ramps up from 0 to 2e-4
# basically stretching before a run so the model doesnt get shocked by big updates at the start
WARMUP_RATIO = 0.03
# cosine = after warmup the learning rate slowly drops to ~0 by the end (follows a cosine curve)
# like slowing your bike down as you get close to where you want to stop
LR_SCHEDULER = "cosine"
# fixed random seed so running it again gives (mostly) the same result
# 3407 is unsloth's default (from that "seed 3407 is all you need" paper lol)
SEED = 3407

# ---- before/after demo ----
# 3 questions from the DEV split (databases the model never sees in training)
# we run these before and after training to compare. can swap in any id from the dev file
DEMO_IDS = ["spider_train_01110", "spider_train_01111", "spider_train_01112"]   # climbing (MQL-0010 dev DB)
# max tokens the model is allowed to write per answer. answers are way shorter, this is just a safety cap
MAX_NEW_TOKENS = 512
# how many dev examples to use for the before/after loss (0 = skip it, saves a few minutes)
DEV_LOSS_SAMPLES = 100

# ---- output ----
RUN_NAME = "qwen25coder3b_lora_r16"   # rename this when you change settings so runs dont overwrite each other
if ON_KAGGLE:
    OUTPUT_DIR = "/kaggle/working/" + RUN_NAME
else:
    OUTPUT_DIR = "/content/outputs/" + RUN_NAME
COLAB_COPY_TO_DRIVE = True   # colab only: also copy the adapter to drive (<drive dir>/../runs/RUN_NAME) so it doesnt vanish

## 2. Install Unsloth (without touching torch)

Tries the safest option first and only loosens it if that fails:
1. keep the installed torch, torchvision, torchaudio **and** numpy;
2. keep only the torch family;
3. let uv choose (torchvision and torchaudio are installed together with torch so they stay matched).

It prints which one worked. If Unsloth is already installed (for example after a kernel restart), it skips.
If a later cell fails with an import error, restart the kernel and run from the top: the install is skipped
the second time and the fresh kernel picks up the new packages.

In [ ]:
import importlib.util
import importlib.metadata
import subprocess
import tempfile

# gets the installed version of a package, or None if it's not installed
# the split("+") removes stuff like "+cu130" from the torch version
def get_version(pkg):
    try:
        v = importlib.metadata.version(pkg)
        return v.split("+")[0]
    except importlib.metadata.PackageNotFoundError:
        return None

# runs "uv pip install ..." and returns (did it work, end of the log)
# using uv instead of normal pip bcoz it's waaay faster
def uv_install(args):
    cmd = ["uv", "pip", "install", "--python", sys.executable, "-q"] + args
    result = subprocess.run(cmd, capture_output=True, text=True)
    log = result.stderr or result.stdout
    log = log[-3000:]   # only keep the end, the full log is huge
    return result.returncode == 0, log

if importlib.util.find_spec("unsloth") is not None:
    # already installed (happens after a kernel restart), no need to do it again
    print("unsloth already installed:", get_version("unsloth"), "- skipping install")
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

    # note down the torch versions kaggle/colab already gave us so we dont break them
    torch_versions = {}
    for p in ["torch", "torchvision", "torchaudio"]:
        torch_versions[p] = get_version(p)
    torch_and_numpy = dict(torch_versions)
    torch_and_numpy["numpy"] = get_version("numpy")

    # try 3 times, each time a bit less strict:
    # 1) dont touch torch stuff OR numpy
    # 2) dont touch torch stuff, numpy is allowed to change
    # 3) just let uv do whatever it wants
    attempts = [
        ("keep torch family + numpy", torch_and_numpy),
        ("keep torch family", torch_versions),
        ("unconstrained", None),
    ]

    worked = False
    log = ""
    for label, keep in attempts:
        args = list(UNSLOTH_SPEC)
        if keep is None:
            # if torch gets upgraded, install torchvision + torchaudio along with it so the versions match
            if torch_versions["torch"]:
                args = args + ["torchvision", "torchaudio"]
        else:
            # write a constraints file ("torch==2.x.x" etc) so uv is NOT allowed to change those
            pins = []
            for p, v in keep.items():
                if v:
                    pins.append(p + "==" + v)
            constraints_file = tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False)
            constraints_file.write("\n".join(pins))
            constraints_file.close()
            args = args + ["-c", constraints_file.name]

        ok, log = uv_install(args)
        if ok:
            print("[ok]", label)
            worked = True
            break
        else:
            print("[failed]", label)
            if log.strip():
                print("   ", log.strip().splitlines()[-1])

    if not worked:
        raise RuntimeError("Unsloth install failed with every option. Check that Internet is On (Kaggle settings). "
                           "Last error:\n" + log)

    # double check torch didnt get changed anyway
    changed = {}
    for p, v in torch_versions.items():
        new_v = get_version(p)
        if v and new_v != v:
            changed[p] = (v, new_v)
    if changed:
        print("torch family changed:", changed)
    else:
        print("torch family unchanged")

### Check the environment

In [ ]:
import importlib
import importlib.metadata

# make sure everything actually imports (pip saying "installed" doesnt mean it works lol)
packages = ["unsloth", "unsloth_zoo", "torch", "transformers", "trl", "peft", "accelerate", "bitsandbytes", "datasets"]
problems = []
for name in packages:
    try:
        importlib.import_module(name)
        # the :14s just pads the name so the versions line up
        print(f"{name:14s} {importlib.metadata.version(name)}")
    except Exception as e:
        problems.append(f"{name}: {type(e).__name__}: {e}")

if len(problems) > 0:
    raise RuntimeError("Some packages don't import. Restart the kernel and run from the top (install will be skipped).\n"
                       + "\n".join(problems))

import torch
# no gpu = no training, stop here
assert torch.cuda.is_available(), "No GPU. Kaggle: Settings → Accelerator → GPU T4 x2. Colab: Runtime → T4 GPU."
cap = torch.cuda.get_device_capability(0)
print("GPU:", torch.cuda.get_device_name(0), "capability", cap, "| torch CUDA", torch.version.cuda)
# compute capability is basically the gpu's "generation number". unsloth needs 7.0 or newer
# T4 = 7.5 (fine), P100 = 6.0 (too old)
assert cap >= (7, 0), "This GPU (e.g. P100) is too old for Unsloth. Switch the accelerator to T4."

## 3. Load data

In [ ]:
import json
import os
import shutil
import glob

# step 1: find where the data files are
if DATA_DIR:
    data_dir = DATA_DIR
elif ON_KAGGLE:
    # kaggle puts datasets somewhere inside /kaggle/input but the exact path depends on the dataset name,
    # so just search for the train file everywhere in there
    hits = sorted(glob.glob(f"/kaggle/input/**/{TRAIN_FILE}", recursive=True))
    assert len(hits) > 0, f"{TRAIN_FILE} not found under /kaggle/input. Add the dataset with 'Add Input'."
    if len(hits) > 1:
        print("several copies found, using the first:", hits)
    data_dir = os.path.dirname(hits[0])
elif COLAB_USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    data_dir = COLAB_DRIVE_DIR
else:
    # no drive, so upload the 2 files by hand
    from google.colab import files
    data_dir = "/content/data"
    os.makedirs(data_dir, exist_ok=True)
    print(f"Upload {TRAIN_FILE} and {DEV_FILE}")
    uploaded = files.upload()
    for name in uploaded:
        with open(os.path.join(data_dir, name), "wb") as f:
            f.write(uploaded[name])
print("data dir:", data_dir)

# step 2: read them. jsonl = one json object per line
def read_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():   # skip empty lines
                rows.append(json.loads(line))
    return rows

train_rows = read_jsonl(os.path.join(data_dir, TRAIN_FILE))
dev_rows = read_jsonl(os.path.join(data_dir, DEV_FILE))
print(f"train: {len(train_rows)}  dev: {len(dev_rows)}")

# step 3: look at one example to make sure the format is what we expect
# each row = {"id": ..., "prompt": [system msg, user msg], "completion": [assistant msg]}
ex = train_rows[0]
roles = []
for m in ex["prompt"]:
    roles.append(m["role"])
assert roles == ["system", "user"], "prompt should be system + user"
assert ex["completion"][0]["role"] == "assistant", "completion should be the assistant's answer"
print(ex["prompt"][1]["content"][-400:])   # only the last 400 chars, the schema part is long
print("--- target ---")
print(ex["completion"][0]["content"])

## 4. Sweep definition

In [ ]:
# every entry = one adapter. anything not listed uses the value from the CONFIG cell above.
# change ONE thing per entry vs the baseline, otherwise you can't tell what helped.
# RUN_NAME must be unique: it's the output folder, and the eval notebook tells adapters apart by it.
# "_q10" = trained WITHOUT the MQL-0010 holdout databases. keep it in every name so these never get mixed up
# with the older adapters (those saw the holdout questions and must not be tested on MQL-0010).
SWEEP = [
    {"RUN_NAME": "q3b_r16_q10"},                                   # baseline (config values)
    {"RUN_NAME": "q3b_r16_ep2_q10", "NUM_EPOCHS": 2},             # train longer
    {"RUN_NAME": "q3b_r16_lr1e-4_q10", "LEARNING_RATE": 1e-4},    # smaller steps
    {"RUN_NAME": "q3b_r32_q10", "LORA_R": 32, "LORA_ALPHA": 64},  # bigger adapter (alpha = 2r)
]
# FINAL MODEL (after the eval notebook picks a winner): set TRAIN_FILE = "sft_full.prompt_completion.jsonl" in the
# CONFIG cell and replace SWEEP with ONE entry: the winner's settings + RUN_NAME ending in "_final_q10".

# ---- MQL-0010 safety check: none of the 18 holdout databases may be in the training data ----
MQL0010_HOLDOUT_DBS = {
    "allergy_1", "behavior_monitoring", "cre_Theme_park", "customers_and_addresses", "e_learning", "game_1",
    "insurance_and_eClaims", "local_govt_and_lot", "mountain_photos", "news_report", "program_share", "race_track",
    "railway", "school_player", "shop_membership", "store_product", "student_1", "workshop_paper"}
def _db(row):
    return row["prompt"][1]["content"].split("\n", 1)[0].replace("Database: ", "").strip()
_leak = sorted({_db(r) for r in train_rows + dev_rows} & MQL0010_HOLDOUT_DBS)
assert not _leak, (f"STOP: training data contains MQL-0010 holdout databases {_leak}. You attached the old approved_v2_1 "
                   "dataset; attach approved_v2_1_mql0010_view instead (and only that one).")
print(f"MQL-0010 check OK: {len(train_rows)} train / {len(dev_rows)} dev rows, no holdout database")

SMOKE_TEST = False   # True = every run trains only 5 steps (checks the whole loop works). False for the real commit.

# these are the knobs a sweep entry may change
SWEEP_KEYS = ["RUN_NAME", "LORA_R", "LORA_ALPHA", "LORA_DROPOUT", "TARGET_MODULES", "NUM_EPOCHS", "MAX_STEPS",
              "BATCH_SIZE", "GRAD_ACCUM", "LEARNING_RATE", "WARMUP_RATIO", "LR_SCHEDULER"]
DEFAULTS = {k: globals()[k] for k in SWEEP_KEYS}

names = [r["RUN_NAME"] for r in SWEEP]
assert len(names) == len(set(names)), "two sweep entries have the same RUN_NAME"
for r in SWEEP:
    bad = [k for k in r if k not in SWEEP_KEYS]
    assert not bad, f"{r['RUN_NAME']}: unknown key(s) {bad} (allowed: {SWEEP_KEYS})"
    if r.get("LORA_R") and not r.get("LORA_ALPHA"):
        print(f"note: {r['RUN_NAME']} changes LORA_R but not LORA_ALPHA (plan says alpha = 2 * r)")

OUTPUT_ROOT = "/kaggle/working" if ON_KAGGLE else "/content/outputs"
os.makedirs(OUTPUT_ROOT, exist_ok=True)
# rough time: ~40 min per epoch of 3B r16 on a T4, + a few minutes per run for loading and dev loss
est = sum((r.get("NUM_EPOCHS", NUM_EPOCHS) * 40 + 6) for r in SWEEP)
print(f"{len(SWEEP)} runs: {names}")
print(f"estimated time: ~{est/60:.1f} h" + ("  (SMOKE_TEST: much less)" if SMOKE_TEST else ""))

## 5. Helpers (same as the single-run notebook)

In [ ]:
# IMPORTANT: import unsloth BEFORE transformers/trl
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only
import torch
import time
import gc
import glob
import inspect
import datetime
import traceback
import subprocess
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

by_id = {}
for r in train_rows + dev_rows:
    by_id[r["id"]] = r
dev_ids = set(r["id"] for r in dev_rows)
for i in DEMO_IDS:
    assert i in dev_ids, f"{i} is not in the dev split; demo questions must come from unseen databases"


def free_gpu():
    # drop the previous run's model + trainer so the next run starts from a clean GPU
    for name in ["trainer", "model"]:
        if name in globals():
            del globals()[name]
    gc.collect()
    torch.cuda.empty_cache()


def load_base():
    # fresh 4-bit base model every run, so no run inherits another run's LoRA
    global model, tokenizer
    free_gpu()
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True)


def chat_ids(messages, add_generation_prompt):
    ids = tokenizer.apply_chat_template(messages, tokenize=True, return_dict=False,
                                        add_generation_prompt=add_generation_prompt)
    return list(ids)


def generate(row):
    input_ids = torch.tensor([chat_ids(row["prompt"], True)], device="cuda")
    with torch.no_grad():
        out = model.generate(input_ids=input_ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    new_tokens = out[0, input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def shape_check(text, row):
    # does NOT check if the query is correct, only Atlas can tell that
    t = text.strip()
    if t.startswith("```"):
        t = t.strip("`")
        if "\n" in t:
            t = t[t.find("\n") + 1:]
    try:
        obj = json.loads(t)
    except Exception:
        return "not valid JSON"
    if not isinstance(obj, dict) or set(obj) != {"collection", "pipeline"}:
        return "JSON, but not exactly {collection, pipeline}"
    if f"- {obj['collection']}\n" not in row["prompt"][1]["content"]:
        return "unknown collection"
    if not isinstance(obj["pipeline"], list):
        return "pipeline is not a list"
    return "ok (valid shape)"


def show_demo(tag):
    FastLanguageModel.for_inference(model)
    results = {}
    for i in DEMO_IDS:
        row = by_id[i]
        q = row["prompt"][1]["content"].split("Question:")[-1].strip().splitlines()[0]
        out = generate(row)
        results[i] = out
        print(f"=== [{tag}] {i}\nQ: {q}\nmodel : {out}\ngold  : {row['completion'][0]['content']}\n"
              f"check : {shape_check(out, row)}\n")
    return results


def dev_loss(n):
    # average loss on the answer tokens only, first n dev rows
    if n <= 0:
        return None
    FastLanguageModel.for_inference(model)
    losses = []
    for row in dev_rows[:n]:
        prompt_ids = chat_ids(row["prompt"], True)
        full_ids = chat_ids(row["prompt"] + row["completion"], False)
        assert full_ids[:len(prompt_ids)] == prompt_ids, "chat template prefix mismatch"
        ids = torch.tensor([full_ids], device="cuda")
        labels = ids.clone()
        labels[0, :len(prompt_ids)] = -100
        with torch.no_grad():
            losses.append(model(input_ids=ids, labels=labels).loss.item())
    return sum(losses) / len(losses)


def build(rows, name):
    texts, n_tok = [], []
    for r in rows:
        t = tokenizer.apply_chat_template(r["prompt"] + r["completion"], tokenize=False)
        texts.append(t)
        n_tok.append(len(tokenizer(t, add_special_tokens=False)["input_ids"]))
    lens = sorted(n_tok)
    keep = [texts[i] for i in range(len(texts)) if n_tok[i] <= MAX_SEQ_LENGTH]
    print(f"{name}: {len(texts)} rows | tokens mean {sum(lens)/len(lens):.0f}, "
          f"p99 {lens[int(0.99*(len(lens)-1))]}, max {lens[-1]} | dropped (> {MAX_SEQ_LENGTH}): {len(texts)-len(keep)}")
    return Dataset.from_dict({"text": keep})

## 6. Base model once ("before"), then train every config

The "before" demo and dev loss are the same for every run (same base model), so they're computed once.

In [ ]:
# ===== BEFORE (base model, once) =====
load_base()
print(torch.cuda.get_device_name(0), f"{torch.cuda.max_memory_reserved()/2**30:.1f} GB reserved after load")
BEFORE = show_demo("before")
LOSS_BEFORE = dev_loss(DEV_LOSS_SAMPLES)
print("dev loss before:", LOSS_BEFORE)
train_ds = build(train_rows, "train")   # same tokenizer every run, so build once

In [ ]:
def train_one(s):
    """Trains one sweep config `s` (full settings dict) from a fresh base model and saves the adapter."""
    global model, trainer
    out_dir = os.path.join(OUTPUT_ROOT, s["RUN_NAME"])
    load_base()
    model = FastLanguageModel.get_peft_model(
        model, r=s["LORA_R"], lora_alpha=s["LORA_ALPHA"], lora_dropout=s["LORA_DROPOUT"],
        target_modules=s["TARGET_MODULES"], bias="none", use_gradient_checkpointing="unsloth", random_state=SEED)
    model.print_trainable_parameters()

    max_steps = 5 if SMOKE_TEST else s["MAX_STEPS"]
    cfg = {
        "output_dir": out_dir, "dataset_text_field": "text", "packing": False,
        "num_train_epochs": s["NUM_EPOCHS"], "max_steps": max_steps,
        "per_device_train_batch_size": s["BATCH_SIZE"], "gradient_accumulation_steps": s["GRAD_ACCUM"],
        "learning_rate": s["LEARNING_RATE"], "lr_scheduler_type": s["LR_SCHEDULER"],
        "optim": "adamw_8bit", "weight_decay": 0.01,
        "fp16": not is_bfloat16_supported(), "bf16": is_bfloat16_supported(),
        "logging_steps": 10, "save_strategy": "steps", "save_steps": 100, "save_total_limit": 2,
        "seed": SEED, "report_to": "none", "dataset_num_proc": 2,
    }
    sft_params = inspect.signature(SFTConfig.__init__).parameters
    cfg["max_length" if "max_length" in sft_params else "max_seq_length"] = MAX_SEQ_LENGTH
    cfg["warmup_ratio" if "warmup_ratio" in sft_params else "warmup_steps"] = s["WARMUP_RATIO"]
    for k in [k for k in cfg if k not in sft_params]:
        print("not supported by this TRL version, dropped:", k)
        del cfg[k]

    FastLanguageModel.for_training(model)
    tr_args = inspect.signature(SFTTrainer.__init__).parameters
    tok_kw = {"processing_class": tokenizer} if "processing_class" in tr_args else {"tokenizer": tokenizer}
    trainer = SFTTrainer(model=model, train_dataset=train_ds, args=SFTConfig(**cfg), **tok_kw)
    trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                      response_part="<|im_start|>assistant\n")
    lab = [t for t in trainer.train_dataset[0]["labels"] if t != -100]
    print("trained-on text:", tokenizer.decode(lab)[:200])

    start = time.time()
    stats = trainer.train()
    minutes = (time.time() - start) / 60
    print(f"[{s['RUN_NAME']}] training took {minutes:.1f} min | train loss {stats.training_loss:.4f} | "
          f"peak GPU {torch.cuda.max_memory_reserved()/2**30:.1f} GB")

    after = show_demo("after " + s["RUN_NAME"])
    loss_after = dev_loss(DEV_LOSS_SAMPLES)
    print(f"[{s['RUN_NAME']}] dev loss before {LOSS_BEFORE}  after {loss_after}")

    model.save_pretrained(out_dir)
    tokenizer.save_pretrained(out_dir)
    run_info = {
        "run_name": s["RUN_NAME"], "model": MODEL_NAME, "max_seq_length": MAX_SEQ_LENGTH,
        "lora": {"r": s["LORA_R"], "alpha": s["LORA_ALPHA"], "dropout": s["LORA_DROPOUT"], "targets": s["TARGET_MODULES"]},
        "train": {"epochs": s["NUM_EPOCHS"], "max_steps": max_steps, "batch": s["BATCH_SIZE"], "grad_accum": s["GRAD_ACCUM"],
                  "lr": s["LEARNING_RATE"], "warmup_ratio": s["WARMUP_RATIO"], "scheduler": s["LR_SCHEDULER"], "seed": SEED},
        "data": {"train_rows_used": len(train_ds), "dev_rows": len(dev_rows), "files": [TRAIN_FILE, DEV_FILE]},
        "results_not_a_benchmark": {"train_loss": stats.training_loss, "train_minutes": round(minutes, 1),
                                    "dev_loss_before": LOSS_BEFORE, "dev_loss_after": loss_after,
                                    "dev_loss_samples": DEV_LOSS_SAMPLES},
        "demo": {i: {"before": BEFORE[i], "after": after[i]} for i in DEMO_IDS},
        "smoke_test": SMOKE_TEST, "sweep": SWEEP_ID,
        "gpu": torch.cuda.get_device_name(0), "platform": "kaggle" if ON_KAGGLE else "other",
        "finished_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    }
    with open(os.path.join(out_dir, "run_info.json"), "w") as f:
        json.dump(run_info, f, indent=1)
    # checkpoints were only a safety net while training; the final adapter is saved, so drop them
    for ck in glob.glob(os.path.join(out_dir, "checkpoint-*")):
        shutil.rmtree(ck, ignore_errors=True)
    return {"train_loss": round(stats.training_loss, 4), "dev_loss_after": loss_after, "train_minutes": round(minutes, 1)}


# ===== SWEEP =====
SWEEP_ID = datetime.datetime.now(datetime.timezone.utc).strftime("sweep_%Y%m%dT%H%MZ")
SUMMARY = []
t0 = time.time()
for idx, run in enumerate(SWEEP, 1):
    s = dict(DEFAULTS)
    s.update(run)
    name = s["RUN_NAME"]
    print(f"\n{'#'*20} [{idx}/{len(SWEEP)}] {name}  ({(time.time()-t0)/60:.0f} min into the sweep) {'#'*20}")
    print({k: s[k] for k in run})
    if os.path.exists(os.path.join(OUTPUT_ROOT, name, "adapter_model.safetensors")):
        print("already trained in this session, skipping")
        SUMMARY.append({"run_name": name, "status": "skipped (exists)"})
        continue
    try:
        res = train_one(s)
        SUMMARY.append({"run_name": name, "status": "ok", **res,
                        "changed": {k: v for k, v in run.items() if k != "RUN_NAME"}})
    except Exception as e:
        # one broken config shouldn't kill the whole night: record it and move on
        traceback.print_exc()
        SUMMARY.append({"run_name": name, "status": f"FAILED: {type(e).__name__}: {e}"[:500]})
    finally:
        free_gpu()
    with open(os.path.join(OUTPUT_ROOT, "sweep_summary.json"), "w") as f:   # rewritten after every run
        json.dump({"sweep": SWEEP_ID, "smoke_test": SMOKE_TEST, "dev_loss_before": LOSS_BEFORE, "runs": SUMMARY}, f, indent=1)

## 7. Summary

In [ ]:
with open(os.path.join(OUTPUT_ROOT, "pip_freeze.txt"), "w") as f:
    f.write(subprocess.run(["pip", "freeze"], capture_output=True, text=True).stdout)

print(f"sweep {SWEEP_ID} | {(time.time()-t0)/60:.0f} min total | dev loss before: {LOSS_BEFORE}\n")
print(f"{'run':40s} {'status':10s} {'train loss':>10s} {'dev loss':>9s} {'min':>6s}")
for r in SUMMARY:
    st = "ok" if r["status"] == "ok" else r["status"][:60]
    dl = r.get("dev_loss_after")
    dl = f"{dl:.4f}" if dl is not None else ""
    print(f"{r['run_name']:40s} {st:10s} {str(r.get('train_loss', '')):>10s} {dl:>9s} {str(r.get('train_minutes', '')):>6s}")
failed = [r["run_name"] for r in SUMMARY if r["status"].startswith("FAILED")]
if failed:
    print("\nFAILED runs (see the traceback above each):", failed)
print("\nLower dev loss = the run worked, NOT that it's more accurate. Rank them with codegen_finetune_eval.ipynb.")
for d in sorted(glob.glob(os.path.join(OUTPUT_ROOT, "*", "adapter_model.safetensors"))):
    print(f"  {os.path.dirname(d)}  ({os.path.getsize(d)/2**20:.0f} MB)")